In [1]:

# BLOQUE 1: Montar Drive y verificar artefactos de Guía 9 (Smart Campus)

from google.colab import drive
drive.mount('/content/drive')

import os

# Carpeta raíz de Smart Campus en Google Drive
BASE = "/content/drive/MyDrive/proyecto_smart_campus"

# Crear carpetas si no existen
for carpeta in ["modelos", "documentos", "datos/procesados", "notebooks"]:
    os.makedirs(f"{BASE}/{carpeta}", exist_ok=True)

print("modelos/:")
print(os.listdir(f"{BASE}/modelos"))
print("\ndocumentos/:")
print(os.listdir(f"{BASE}/documentos"))

# Verificación de artefacto crítico de Guía 9
ruta_g9 = f"{BASE}/modelos/modelo_candidato_guia09.joblib"
print(f"\n{'OK' if os.path.exists(ruta_g9) else 'WARNING '} .joblib de Guía 9: {'encontrado' if os.path.exists(ruta_g9) else 'NO encontrado (se reconstruirá de forma alternativa)'}")


Mounted at /content/drive
modelos/:
[]

documentos/:
[]

WARNING  .joblib de Guía 9: NO encontrado (se reconstruirá de forma alternativa)


In [ ]:

# BLOQUE 2: Importaciones y configuración global
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, sys, time, joblib

# Herramientas NUEVAS de la Guía 10
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, ParameterGrid

# Métricas de Guía 9 y Baseline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("Semilla:", RANDOM_STATE)



In [ ]:

# BLOQUE 3: Carga y reconstrucción del dataset (Smart Campus)

RUTA_DATASET = f"{BASE}/datos/procesados/dataset_limpio_Smart_Campus.csv"

if os.path.exists(RUTA_DATASET):
    df_raw = pd.read_csv(RUTA_DATASET)
    df_raw['fecha'] = pd.to_datetime(df_raw['fecha'])

    # Agrupación diaria (idéntico a la Guía 9)
    df = df_raw.groupby('fecha').agg(
        estudiantes_presentes=('estudiantes_presentes', 'sum'),
        horas_uso_promedio=('horas_uso', 'mean'),
        porcentaje_ocupacion_medio=('porcentaje_ocupacion', 'mean')
    ).reset_index().sort_values('fecha').reset_index(drop=True)

    # Ingeniería de características
    df['dia_semana'] = df['fecha'].dt.dayofweek
    df['mes'] = df['fecha'].dt.month
    df["lag_1"] = df["estudiantes_presentes"].shift(1)
    df["lag_7"] = df["estudiantes_presentes"].shift(7)
    df["media_movil_7"] = df["estudiantes_presentes"].shift(1).rolling(7).mean()

    df_modelo = df.dropna().copy()
    print(f"Dataset cargado desde Drive: {df_modelo.shape[0]} filas, {df_modelo.shape[1]} columnas")
else:
    raise FileNotFoundError(f"No se encontró el dataset en {RUTA_DATASET}. Asegúrate de haberlo subido en la Guía 9.")

df_modelo.head(3)


In [ ]:

# BLOQUE 4: Variables (Features) y división cronológica 80/20 + Guardián

features = [
    "dia_semana", "mes",
    "lag_1", "lag_7", "media_movil_7"
]
X = df_modelo[features]
y = df_modelo["estudiantes_presentes"]

corte = int(len(df_modelo) * 0.80)
X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]
fecha_test = df_modelo["fecha"].iloc[corte:]

# Guardián del experimento (verifica orden cronológico estricto)
assert X_train.index.max() < X_test.index.min(), "¡El split no respeta el orden temporal!"

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Última fecha de train: {df_modelo['fecha'].iloc[corte-1].date()}")
print(f"Primera fecha de test: {df_modelo['fecha'].iloc[corte].date()}")



In [ ]:

# BLOQUE 5: Cargar modelo de Guía 9 (con respaldo) y calcular referencias

ruta_g9 = f"{BASE}/modelos/modelo_candidato_guia09.joblib"

if os.path.exists(ruta_g9):
    modelo_guia9 = joblib.load(ruta_g9)
    print("OK Modelo de Guía 9 cargado desde Drive")
else:
    from sklearn.ensemble import RandomForestRegressor
    modelo_guia9 = RandomForestRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
    print("WARNING .joblib no encontrado. Modelo de referencia reconstruido (RandomForestRegressor).")

# Métricas de referencia en el mismo conjunto de prueba (holdout)
pred_guia9 = modelo_guia9.predict(X_test)
mae_guia9  = mean_absolute_error(y_test, pred_guia9)
rmse_guia9 = mean_squared_error(y_test, pred_guia9) ** 0.5
r2_guia9   = r2_score(y_test, pred_guia9)

# Baseline como referencia mínima absoluta
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
pred_base = baseline.predict(X_test)
mae_base  = mean_absolute_error(y_test, pred_base)
rmse_base = mean_squared_error(y_test, pred_base) ** 0.5
r2_base   = r2_score(y_test, pred_base)

print("\nReferencias de rendimiento:")
print(f"   Baseline → MAE: {mae_base:,.2f} | RMSE: {rmse_base:,.2f} | R²: {r2_base:.4f}")
print(f"   Guía 9   → MAE: {mae_guia9:,.2f} | RMSE: {rmse_guia9:,.2f} | R²: {r2_guia9:.4f}")


In [ ]:

# BLOQUE 6: Pipeline — El escudo anti-fuga (Anti-Leakage)

pipeline = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("modelo", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=1   # 1 núcleo: evita conflictos de paralelismo anidado con GridSearchCV n_jobs=-1
    ))
])

print(pipeline)


In [ ]:

# BLOQUE 7: Definición del espacio de hiperparámetros (Grid)

param_grid = {
    "modelo__n_estimators":     [100, 200],
    "modelo__max_depth":        [5, 10, None],
    "modelo__min_samples_leaf": [1, 3, 5],
    "modelo__max_features":     [0.7, 1.0]
}

combinaciones = list(ParameterGrid(param_grid))
print(f"Combinaciones totales: {len(combinaciones)}")
print(f"Entrenamientos esperados (combinaciones × folds): {len(combinaciones) * 4}")


In [ ]:

# BLOQUE 8: Configurar GridSearchCV con TimeSeriesSplit
cv = TimeSeriesSplit(n_splits=4)

print("Folds de validación temporal (estrictamente sobre X_train):")
for fold, (idx_tr, idx_val) in enumerate(cv.split(X_train), 1):
    print(f"  Fold {fold}: train {idx_tr[0]}–{idx_tr[-1]}  |  val {idx_val[0]}–{idx_val[-1]}")

busqueda = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",   # MAE expresado con signo negativo
    cv=cv,
    n_jobs=-1,                           # Paraleliza el procesamiento
    refit=True,                          # Reentrena el mejor modelo con todo X_train
    return_train_score=True,             # Necesario para calcular la Brecha_MAE
    error_score="raise",
    verbose=1
)

print("\nOK GridSearchCV configurado. Listo para ejecutar .fit()")

In [ ]:

# BLOQUE 9: Ejecutar la búsqueda (SOLO sobre X_train)

inicio = time.perf_counter()
busqueda.fit(X_train, y_train)   # <-- NUNCA debemos de pasar X, y completos aquí
tiempo_total = time.perf_counter() - inicio

print("\n" + "=" * 55)
print(f"Tiempo total de búsqueda: {tiempo_total:.2f} s")
print(f"Mejores parámetros encontrados: {busqueda.best_params_}")
print(f"MAE real en validación cruzada: {-busqueda.best_score_:,.2f}")
print("=" * 55)


In [ ]:


# BLOQUE 10: Convertir cv_results_ en una tabla analizable

resultados = pd.DataFrame(busqueda.cv_results_)

columnas = ["rank_test_score", "mean_test_score", "std_test_score",
            "mean_train_score", "mean_fit_time", "params"]
tabla_grid = resultados[columnas].copy()

tabla_grid["MAE_validacion"]    = -tabla_grid["mean_test_score"]
tabla_grid["MAE_entrenamiento"] = -tabla_grid["mean_train_score"]
tabla_grid["Brecha_MAE"]        = tabla_grid["MAE_validacion"] - tabla_grid["MAE_entrenamiento"]

tabla_grid = tabla_grid.sort_values("rank_test_score").reset_index(drop=True)
tabla_grid.head(10)


In [ ]:

# BLOQUE 11: Gráfico del Top 10 de configuraciones

top = tabla_grid.nsmallest(10, "MAE_validacion").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(top)), top["MAE_validacion"],
       yerr=top["std_test_score"], capsize=4, edgecolor="black")
ax.set_xticks(range(len(top)))
ax.set_xticklabels([f"C{i+1}" for i in range(len(top))])
ax.set_ylabel("MAE de validación (Estudiantes)")
ax.set_title("Top 10 configuraciones — GridSearchCV (Smart Campus)", fontweight="bold")
ax.grid(alpha=0.3, axis="y", linestyle="--")
plt.tight_layout()
plt.show()


In [ ]:


# BLOQUE 12: Evaluar el pipeline optimizado en el Holdout final

modelo_optimizado = busqueda.best_estimator_   # Ya reentrenado con todo X_train
pred_test = modelo_optimizado.predict(X_test)   # Primera y única vez que se toca

mae  = mean_absolute_error(y_test, pred_test)
rmse = mean_squared_error(y_test, pred_test) ** 0.5
r2   = r2_score(y_test, pred_test)

print("Modelo optimizado — Holdout final:")
print(f"   MAE:  {mae:,.2f}")
print(f"   RMSE: {rmse:,.2f}")
print(f"   R²:   {r2:.4f}")

In [ ]:

# BLOQUE 13: Tabla comparativa (Baseline vs Guía 9 vs Guía 10)

comparacion_guia10 = pd.DataFrame([
    {"Modelo": "Baseline",       "MAE": mae_base,  "RMSE": rmse_base,  "R2": r2_base},
    {"Modelo": "Guía 9",         "MAE": mae_guia9, "RMSE": rmse_guia9, "R2": r2_guia9},
    {"Modelo": "Guía 10 (opt.)", "MAE": mae,       "RMSE": rmse,       "R2": r2}
])

comparacion_guia10["Mejora_vs_baseline_%"] = (
    (mae_base - comparacion_guia10["MAE"]) / mae_base * 100
)
comparacion_guia10["Mejora_vs_G9_%"] = (
    (mae_guia9 - comparacion_guia10["MAE"]) / mae_guia9 * 100
)

comparacion_guia10.round(2)

In [ ]:

# BLOQUE 14: Top de mayores errores absolutos del modelo optimizado

errores = pd.DataFrame({
    "fecha":    fecha_test.values,
    "real":     y_test.values,
    "predicho": pred_test
})
errores["error"]     = errores["real"] - errores["predicho"]
errores["error_abs"] = errores["error"].abs()

print("Estadísticas de error:")
print(f"  Error promedio: {errores['error'].mean():,.2f}")
print(f"  Error mínimo:  {errores['error'].min():,.2f}")
print(f"  Error máximo:  {errores['error'].max():,.2f}")

errores.nlargest(10, "error_abs").round(2)

In [ ]:

# BLOQUE 15: Guardar artefactos reproducibles en Google Drive

resultados.to_csv(f"{BASE}/documentos/gridsearch_resultados_guia10.csv",
                  index=False, encoding="utf-8")

comparacion_guia10.to_csv(f"{BASE}/documentos/comparacion_antes_despues_guia10.csv",
                          index=False, encoding="utf-8")

errores.to_csv(f"{BASE}/datos/procesados/predicciones_guia10.csv",
               index=False, encoding="utf-8")

joblib.dump(modelo_optimizado, f"{BASE}/modelos/modelo_optimizado_guia10.joblib")

print("OK Artefactos guardados exitosamente en Drive")

In [ ]:

# BLOQUE 16: Metadatos y trazabilidad

modelo_interno = modelo_optimizado.named_steps["modelo"]

metadata = {
    "fecha_ejecucion":      pd.Timestamp.now().isoformat(),
    "python":               sys.version.split()[0],
    "scikit_learn":         sklearn.__version__,
    "random_state":         RANDOM_STATE,
    "n_combinaciones":      len(combinaciones),
    "n_folds":              cv.get_n_splits(),
    "n_train":              len(X_train),
    "n_test":               len(X_test),
    "tiempo_busqueda_s":    round(tiempo_total, 2),
    "mejores_parametros":   str(busqueda.best_params_),
    "modelo_candidato_g9":  type(modelo_guia9).__name__,
    "modelo_optimizado":    type(modelo_interno).__name__,
    "mae_guia9":            round(mae_guia9, 2),
    "mae_optimizado":       round(mae, 2),
    "mejora_vs_g9_%":       round(comparacion_guia10["Mejora_vs_G9_%"].iloc[2], 2)
}

pd.DataFrame([metadata]).to_csv(
    f"{BASE}/documentos/metadata_guia10.csv",
    index=False, encoding="utf-8"
)

for k, v in metadata.items():
    print(f"  {k}: {v}")

In [ ]:

# BLOQUE 17: Auditoría final y empaquetado para descarga en ZIP

archivos_esperados = [
    f"{BASE}/documentos/gridsearch_resultados_guia10.csv",
    f"{BASE}/documentos/comparacion_antes_despues_guia10.csv",
    f"{BASE}/documentos/metadata_guia10.csv",
    f"{BASE}/datos/procesados/predicciones_guia10.csv",
    f"{BASE}/modelos/modelo_optimizado_guia10.joblib",
]

print("Auditoría final:\n")
todo_ok = True
for ruta in archivos_esperados:
    existe = os.path.exists(ruta)
    peso = f"{os.path.getsize(ruta)/1024:.1f} KB" if existe else "—"
    icono = "OK" if existe else "FAILED"
    print(f"  {icono} {os.path.basename(ruta):<45} {peso}")
    todo_ok &= existe

print("\n" + ("GOOD!!! Todos los artefactos están presentes." if todo_ok
             else "WARNING!!! Faltan artefactos. Revisa las celdas previas."))

import zipfile
from google.colab import files

nombre_zip = "artefactos_guia10.zip"
with zipfile.ZipFile(nombre_zip, 'w') as zipf:
    for ruta in archivos_esperados:
        if os.path.exists(ruta):
            zipf.write(ruta, arcname=os.path.basename(ruta))

files.download(nombre_zip)
print("OK Descarga del paquete .zip iniciada correctamente.")